# Data Drift Guardian — демонстрация

Полный цикл работы системы на демонстрационном датасете кредитного скоринга с искусственно внесённым дрейфом:
генерация данных → анализ (`analyze`) → интерпретация метрик → графики → экспорт HTML/JSON.

Все данные генерируются с фиксированным seed, поэтому результаты воспроизводимы.

In [1]:
import sys
sys.path.insert(0, "..")  # запуск из папки notebooks без установки пакета

import pandas as pd
from drift_guardian import DriftConfig, analyze
from drift_guardian.demo import SCENARIOS, make_demo
from drift_guardian.plots import column_summary_frame, style_severity, numeric_distribution_figure, categorical_distribution_figure
from drift_guardian.html_report import save_html_report, report_to_json

pd.set_option("display.width", 160)
for name, description in SCENARIOS.items():
    print(f"{name:14s} — {description}")

no_drift       — Без дрейфа — контрольный батч из того же распределения
mean_shift     — Сдвиг среднего — аудитория постарела, доходы выросли на 25%
missing_surge  — Рост пропусков — доля пустого дохода выросла на ~11 п.п.
new_category   — Новая категория — появился канал «партнёрская сеть»
concept_drift  — Концептуальный дрейф — признаки стабильны, но доля дефолтов выросла с ~16% до ~40%
mixed          — Всё вместе — плюс рост разброса сумм кредита


## 1. Данные

Эталон — 20 000 заявок на кредит, на которых «обучалась модель». Текущий батч — 5 000 заявок из продакшна.
Колонка `target` (1 — дефолт) порождена логистической моделью от признаков, поэтому в сценарии
`concept_drift` меняется связь «признаки → дефолт», а сами признаки — нет.

In [2]:
reference, current = make_demo("mixed", ref_rows=20_000, cur_rows=5_000, seed=42)
print(reference.shape, current.shape)
reference.head()

(20000, 9) (5000, 9)


,age,income,loan_amount,credit_history_years,num_dependents,employment_type,region,channel,target
0,21.0,69728.0,203308.0,4.3,3,наёмный,Москва,мобильное приложение,0
1,47.0,35597.0,57049.0,19.2,1,наёмный,Миллионники,офис,0
2,37.0,87136.0,45884.0,10.6,1,наёмный,Москва,мобильное приложение,1
3,32.0,95825.0,204904.0,7.3,3,наёмный,Москва,мобильное приложение,0
4,35.0,22572.0,18076.0,6.0,3,наёмный,Москва,мобильное приложение,1


## 2. Сценарий «всё вместе» (`mixed`)

Указываем целевую переменную — она анализируется отдельным блоком и не участвует в adversarial validation.

In [3]:
config = DriftConfig(target_column="target")
report = analyze(reference, current, config)

print("Статус:", report["overall_severity"].upper())
print("Рекомендация:", report["recommendation"])
print()
for alert in report["alerts"]:
    print("-", alert)

Статус: CRITICAL
Рекомендация: Зафиксирован критический дрейф, рекомендуется переобучение модели.

- ВНИМАНИЕ: Доля пропусков в 'income' выросла с 7.7% до 19.3% (+11.6 п.п.).
- КРИТИЧНО: 16.8% значений 'loan_amount' вне диапазона эталона [5857, 1.23218e+06].
- КРИТИЧНО: В 'channel' появились категории, не предусмотренные эталоном: ['партнёрская сеть'] — 11.0% строк текущего батча.
- ВНИМАНИЕ: дрейф целевой переменной 'target' — chi2=429 (p<1e-16), jensen_shannon=0.13.
- КРИТИЧНО: дрейф по признаку 'age' — ks=0.239 (p<1e-16), psi=0.337, jensen_shannon=0.243, wasserstein_norm=0.605.
- ВНИМАНИЕ: дрейф по признаку 'income' — ks=0.185 (p<1e-16), psi=0.193, jensen_shannon=0.185, wasserstein_norm=0.465.
- КРИТИЧНО: дрейф по признаку 'loan_amount' — ks=0.221 (p<1e-16), psi=0.388, jensen_shannon=0.26, wasserstein_norm=0.428.
- КРИТИЧНО: дрейф по признаку 'channel' — chi2=2.25e+03 (p<1e-16), psi=0.782, jensen_shannon=0.238.
- КРИТИЧНО: adversarial validation различает выборки (ROC-AUC=0.845); си

### Сводка по признакам

Худшие признаки сверху. `KS / χ²` — статистика p-value-теста, `PSI`, `JS`, `Вассерштейн` — метрики размера эффекта.

In [4]:
style_severity(column_summary_frame(report))

,признак,тип,статус,PSI,JS,Вассерштейн (норм.),KS / χ²,p-value
0,channel,категориальный,критично,0.781800,0.238400,nan,2248.780000,0.000000
1,loan_amount,числовой,критично,0.388000,0.260000,0.427600,0.221300,0.000000
2,age,числовой,критично,0.336800,0.242700,0.604900,0.238900,0.000000
3,income,числовой,внимание,0.193100,0.184800,0.465200,0.185500,0.000000
4,credit_history_years,числовой,в норме,0.000900,0.012800,0.012900,0.008800,0.909902
5,employment_type,категориальный,в норме,0.000400,0.008600,nan,1.652600,0.647514
6,num_dependents,категориальный,в норме,0.000300,0.007800,nan,1.357100,0.715621
7,region,категориальный,в норме,0.000200,0.006600,nan,0.961700,0.810527


### Распределения

Синий — эталон, оранжевый — текущий батч. У `age` виден сдвиг вправо (аудитория постарела),
у `channel` — новая категория «партнёрская сеть».

In [5]:
numeric_distribution_figure(reference["age"], current["age"], title="age").show()
categorical_distribution_figure(reference["channel"], current["channel"], title="channel").show()

### Adversarial validation

LightGBM учится отличать эталон от батча. ROC-AUC ≈ 0.5 — выборки неразличимы; чем выше — тем сильнее
изменилась совместная структура признаков. Важности показывают, что именно изменилось.

In [6]:
adv = report["adversarial"]
print(f"ROC-AUC = {adv['roc_auc']:.3f} ({adv['severity']}), бэкенд: {adv['backend']}")
pd.DataFrame(adv["top_features"])

ROC-AUC = 0.845 (critical), бэкенд: lightgbm


,feature,importance
0,loan_amount,0.4220
1,income,0.2307
2,channel,0.1532
3,age,0.1431
4,credit_history_years,0.0324
5,num_dependents,0.0063
6,employment_type,0.0062
7,region,0.0060


## 3. Все сценарии

Один и тот же эталон, разные батчи. Обратите внимание на `no_drift` (ничего не найдено) и `concept_drift`
(признаки стабильны, но целевая переменная изменилась).

In [7]:
rows = []
for name in SCENARIOS:
    ref, cur = make_demo(name, 20_000, 5_000, seed=42)
    r = analyze(ref, cur, config)
    rows.append({
        "сценарий": name,
        "итог": r["overall_severity"],
        "таргет": r["target_drift"]["severity"],
        "критичных признаков": sum(c["severity"] == "critical" for c in r["columns"]),
        "DQ-алертов": len(r["data_quality"]),
        "adversarial AUC": r["adversarial"]["roc_auc"],
    })
pd.DataFrame(rows)

,сценарий,итог,таргет,критичных признаков,DQ-алертов,adversarial AUC
0,no_drift,ok,ok,0,0,0.5030
1,mean_shift,critical,ok,1,0,0.7561
2,missing_surge,warning,ok,0,1,0.5532
3,new_category,critical,ok,1,1,0.5627
4,concept_drift,critical,critical,0,0,0.5204
5,mixed,critical,warning,3,3,0.8451


## 4. Концептуальный дрейф крупным планом

В сценарии `concept_drift` входные данные не изменились, но доля дефолтов при тех же заявках выросла
с ~16 % до ~40 %. Поколоночные тесты признаков молчат, adversarial validation молчит, а блок целевой переменной
поднимает тревогу — именно этот случай был бы невидим для мониторинга «только по X».

In [8]:
ref_c, cur_c = make_demo("concept_drift", 20_000, 5_000, seed=42)
r = analyze(ref_c, cur_c, config)
print("Итог:", r["overall_severity"], "| adversarial AUC:", r["adversarial"]["roc_auc"])
print("Рекомендация:", r["recommendation"])
print("Доля дефолтов: эталон", round(ref_c["target"].mean(), 3), "→ батч", round(cur_c["target"].mean(), 3))
pd.DataFrame(r["target_drift"]["tests"])[["name", "statistic", "p_value", "severity"]]

Итог: critical | adversarial AUC: 0.5204
Рекомендация: Признаки стабильны, но распределение целевой переменной изменилось: вероятен концептуальный дрейф. Рекомендуется переобучение модели на свежих размеченных данных.
Доля дефолтов: эталон 0.154 → батч 0.402


,name,statistic,p_value,severity
0,chi2,1528.5269,0.0,warning
1,psi,0.3265,NaN,critical
2,jensen_shannon,0.2392,NaN,critical


## 4b. Мониторинг во времени

В продакшене данные приходят потоком. Режим временного ряда режет поток по колонке даты
и прогоняет каждый период против эталона: видно, когда дрейф начался и как он нарастает.
Демо-поток: аудитория постепенно стареет и богатеет, с 5-го месяца растут пропуски,
с 6-го появляется новый канал, с 7-го — концептуальный дрейф по дефолтам.

In [9]:
from drift_guardian import run_timeline_from_frame
from drift_guardian.demo import make_timeline_demo
from drift_guardian.plots import timeline_frame, timeline_psi_figure, timeline_severity_figure

reference_t, stream = make_timeline_demo(n_periods=8, rows_per_period=2_000, seed=42)
timeline = run_timeline_from_frame(reference_t, stream, "date", "M", config, compare_previous=True)
timeline_frame(timeline)

,период,строк,статус,к предыдущему,таргет,critical,warning,adversarial AUC,алертов
0,2026-01,2000,в норме,—,в норме,0,0,0.5062,0
1,2026-02,2000,в норме,в норме,в норме,0,0,0.5302,0
2,2026-03,2000,внимание,в норме,в норме,0,0,0.5677,1
3,2026-04,2000,внимание,в норме,в норме,0,1,0.5979,2
4,2026-05,2000,критично,внимание,в норме,0,2,0.6738,4
5,2026-06,2000,критично,критично,в норме,2,1,0.8374,6
6,2026-07,2000,критично,критично,в норме,2,1,0.7392,6
7,2026-08,2000,критично,внимание,внимание,2,1,0.7772,7


In [10]:
timeline_severity_figure(timeline, title="Относительно эталона: накопленный дрейф").show()
timeline_severity_figure(timeline, title="Относительно предыдущего периода: скачки", source="previous").show()
timeline_psi_figure(timeline).show()

## 4b′. Разрез по сегментам

Дрейф часто локален. Сдвинем возраст только для региона «Москва»: в объединённых данных сдвиг
размывается до «внимания», а внутри сегмента виден критический дрейф.

In [11]:
import numpy as np
from drift_guardian.plots import segment_frame, segment_heatmap_figure

local = current.copy()
msk = local["region"] == "Москва"
local.loc[msk, "age"] = np.clip(local.loc[msk, "age"] + 10, 18, 85)
r_seg = analyze(reference, local, DriftConfig(target_column="target", segment_column="region"))
print("Итог:", r_seg["overall_severity"], "|", r_seg["recommendation"])
segment_frame(r_seg["segments"])

Итог: critical | Зафиксирован критический дрейф, рекомендуется переобучение модели.


,сегмент,строк (эталон / батч),доля батча,статус,critical,warning,первый алерт
0,Миллионники,6062 / 1520,0.3040,критично,3,1,ВНИМАНИЕ: Доля пропусков в 'income' выросла с ...
1,Прочие,5975 / 1517,0.3034,критично,4,0,ВНИМАНИЕ: Доля пропусков в 'income' выросла с ...
2,Москва,5027 / 1225,0.2450,критично,4,0,ВНИМАНИЕ: Доля пропусков в 'income' выросла с ...
3,Санкт-Петербург,2936 / 738,0.1476,критично,3,1,ВНИМАНИЕ: Доля пропусков в 'income' выросла с ...


In [12]:
segment_heatmap_figure(r_seg["segments"]).show()

## 4c. Малые батчи

PSI и JS смещены вверх на малых выборках. Система оценивает шумовой уровень метрики
бутстрепом и поднимает порог warning, если батч слишком мал, а колонку помечает `underpowered`.

In [13]:
small = current.sample(150, random_state=0)
r_small = analyze(reference, small, config)
psi_age = next(t for c in r_small["columns"] if c["column"] == "age" for t in c["tests"] if t["name"] == "psi")
print("Итог на 150 строках:", r_small["overall_severity"])
print("PSI age =", psi_age["statistic"], "| шумовой уровень (99%) =", psi_age["details"]["noise_floor"],
      "| порог:", psi_age["threshold"])
print("Колонки с недостаточной мощностью:", r_small["meta"]["underpowered_columns"])

Итог на 150 строках: critical
PSI age = 0.4566 | шумовой уровень (99%) = 0.1433 | порог: warning >= 0.143, critical >= 0.2 (порог поднят до шумового уровня: батч мал)
Колонки с недостаточной мощностью: ['age', 'income', 'loan_amount', 'credit_history_years', 'num_dependents', 'employment_type', 'region']


## 4d. Реальные данные и вырожденные распределения

Синтетика не заменяет живых данных, поэтому система прогнана на трёх открытых датасетах OpenML
разной формы (`scripts/real_data_gallery.py`; HTML-отчёты и JSON лежат в `examples/real/`):

- **adult, случайный сплит** — контроль на ложные тревоги: 14 признаков, 49 тысяч строк;
- **adult, батч «старше 50»** — контролируемый ковариатный сдвиг при стабильной доле таргета;
- **credit-g, 700 против 300 строк** — маленькая широкая таблица (20 признаков);
- **electricity, начало ряда против конца** — настоящий временной дрейф с разрезом по дням недели.

Сводка ниже читает сохранённые JSON-отчёты, поэтому работает без сети. Пересчитать галерею:
`SSL_CERT_FILE=$(python -c "import certifi; print(certifi.where())") python scripts/real_data_gallery.py`.

In [14]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "real").exists())
rows = []
for path in sorted((root / "examples" / "real").glob("*.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    drifted = [c["column"] for c in r["columns"] if c["severity"] != "ok"]
    rows.append({
        "кейс": path.stem,
        "строк (эталон / батч)": f'{r["meta"]["reference_rows"]} / {r["meta"]["current_rows"]}',
        "признаков": len(r["columns"]),
        "итог": r["overall_severity"],
        "с дрейфом": ", ".join(drifted[:5]) + (" …" if len(drifted) > 5 else "") or "—",
        "таргет": (r.get("target_drift") or {}).get("severity", "—"),
        "adv. AUC": (r.get("adversarial") or {}).get("roc_auc"),
        "сегментов": len(r.get("segments") or []),
    })
pd.DataFrame(rows)

,кейс,строк (эталон / батч),признаков,итог,с дрейфом,таргет,adv. AUC,сегментов
0,adult_age_over_50,30000 / 9808,14,critical,"age, education-num, workclass, education, mari...",ok,1.0000,2
1,adult_random_split,30000 / 18842,14,ok,—,ok,0.5006,2
2,credit_g_small,700 / 300,20,ok,—,ok,0.5166,0
3,electricity_begin_vs_end,15000 / 15000,8,critical,"date, nswprice, nswdemand, vicprice, vicdemand …",ok,1.0000,7


In [15]:
# Два дефекта, которые нашлись только на реальных данных, воспроизводим на игрушечном примере:
# признак, который в эталоне не менялся, а в батче «ожил», и счётчик (время), у которого
# диапазоны эталона и батча не пересекаются.
import numpy as np
import pandas as pd

from drift_guardian.narrative import card_metric, explain_column, issues_by_column

rng = np.random.default_rng(0)
ref_d = pd.DataFrame({
    "flag": np.zeros(3_000),                    # константа в эталоне
    "counter": np.arange(3_000, dtype=float),   # счётчик или время
    "x": rng.normal(0, 1, 3_000),
})
cur_d = pd.DataFrame({
    "flag": rng.uniform(1, 2, 1_500),           # в батче признак разбросан
    "counter": np.arange(3_000, 4_500, dtype=float),
    "x": rng.normal(0, 1, 1_500),
})
report_d = analyze(ref_d, cur_d, DriftConfig(adversarial_enabled=False))
issues_d = issues_by_column(report_d)
for col in report_d["columns"]:
    label, value, *_ = card_metric(col, issues_d.get(col["column"], []), DriftConfig().thresholds)
    print(f'{col["column"]:8} {col["severity"]:9} {explain_column(col, issues_d.get(col["column"], []))}'
          f'  |  полоса карточки: {label} {value:.3f}')
print()
for issue in report_d["data_quality"]:
    if issue["check"] == "disjoint_ranges":
        print(issue["message"])

flag     critical  Сдвиг на 2.06σ, эталон почти константен, 100% значений вне диапазона  |  полоса карточки: вне диапазона 1.000
counter  critical  Сдвиг на 2.60σ, 100% значений вне диапазона  |  полоса карточки: PSI 8.281
x        ok        Без существенных изменений (JS 0.03)  |  полоса карточки: PSI 0.007

Диапазоны 'flag' в эталоне и батче не пересекаются ([0, 0] против [1.00025, 1.99957]). Если это время, счётчик или идентификатор — исключите колонку из анализа; иначе батч целиком вышел за пределы эталона.
Диапазоны 'counter' в эталоне и батче не пересекаются ([0, 2999] против [3000, 4499]). Если это время, счётчик или идентификатор — исключите колонку из анализа; иначе батч целиком вышел за пределы эталона.


Что здесь важно. У `flag` PSI и JS вырождаются в ноль (в эталоне один бин), а сдвиг в «σ эталона»
раньше превращался в число вида 4·10¹⁶: теперь при константном эталоне расстояние Вассерштейна
нормируется на разброс объединённой выборки, карточка показывает долю значений вне диапазона вместо
бесполезного PSI 0.000 и пишет «эталон почти константен». Для `counter` система замечает, что
диапазоны не пересекаются, и подсказывает исключить колонку, если это время или идентификатор,
не глуша само обнаружение. В датасете electricity именно так вели себя `vicprice`, `transfer` и `date`.

## 5. Пороги и ложные срабатывания

p-value-тесты (KS, χ²) на больших выборках находят значимость почти всегда, а на одинаковых распределениях
ложно срабатывают с частотой α. Поэтому в системе действует двухключевое правило: p-value-тест засчитывается,
только если метрики размера эффекта тоже превысили порог. Проверим на батче без дрейфа, как часто KS был бы
«значим» без этого правила.

In [16]:
ref_n, cur_n = make_demo("no_drift", 20_000, 5_000, seed=7)
r = analyze(ref_n, cur_n, config)
significant = [c["column"] for c in r["columns"] if any(t.get("details", {}).get("significant") for t in c["tests"])]
print("Итог по двухключевому правилу:", r["overall_severity"])
print("Колонки, где p-value-тест был статистически значим:", significant or "нет")

Итог по двухключевому правилу: ok
Колонки, где p-value-тест был статистически значим: нет


## 6. Экспорт

HTML-отчёт с интерактивными графиками и JSON по выходному контракту.

In [17]:
from pathlib import Path
out = Path("output"); out.mkdir(exist_ok=True)
path = save_html_report(out / "drift_report_mixed.html", report, reference, current, plotlyjs="cdn")
(out / "drift_report_mixed.json").write_text(report_to_json(report), encoding="utf-8")
print("HTML:", path, f"({path.stat().st_size // 1024} КБ)")
print("Ключи JSON:", list(report.keys()))

HTML: output/drift_report_mixed.html (144 КБ)
Ключи JSON: ['overall_severity', 'recommendation', 'alerts', 'schema', 'data_quality', 'columns', 'adversarial', 'meta', 'target_drift', 'prediction_drift', 'segments']


## Выводы

- Система различает четыре класса проблем: дрейф признаков, проблемы качества данных, многомерный сдвиг
  (adversarial validation) и концептуальный дрейф по целевой переменной.
- Пороги метрик размера эффекта откалиброваны друг относительно друга, p-value-тесты не шумят на больших данных
  благодаря двухключевому правилу, а на малых батчах пороги поднимаются до шумового уровня.
- Режим временного ряда показывает, когда дрейф начался и как нарастает.
- Тот же анализ доступен из дашборда Streamlit (`streamlit run app/streamlit_app.py`) и из командной строки
  (`drift-guardian -r reference.csv -c current.csv --target target --html report.html`).